Detta är koden till chatbotten som används i inlämningsuppgiften

## Importer

In [ ]:
from pypdf import PdfReader
from pathlib import Path
import re
from google import genai
import os
from google.genai import types
from dotenv import load_dotenv
import json
import time
import numpy as np


In [ ]:
load_dotenv()

True

In [ ]:
client = genai.Client(api_key=os.getenv("API_KEY"))

# RAG-delen

## Inläsning av dokumenten

In [ ]:
# Läser in alla pdferna som ligger i mappen "Dokument till RAG" i mitt repository
folder_path = Path("Dokumenten till RAG")
pdf_files = list(folder_path.glob("*.pdf"))

documents = []

for pdf in folder_path.glob("*.pdf"): #läser in alla alla dokument som slutar med pdf
    reader = PdfReader(pdf)

    text = ""
    for page in reader.pages[3:]: # Väljer att inte läsa in framsidan och orelevanta sidor i början
        page_text = page.extract_text()
        # Lägger bara till om det verkligen finns någon text på sidan
        if page_text:
            text += page_text + "\n"
    
    # sparar från vilket dokument texten kommer ifrån
    documents.append({
        "source": pdf.name,
        "text": text
    })


Vid steget kontroll av embeddingsen så upptäckte jag att textkvaliten var dålig då det var radbrytningar mitt i ord i pdfen så därför lägger jag till detta steget som ska städa texten för att underlätta embeddningsen i senare skede. 

In [ ]:
def clean_text(text):
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # ta bort PDF-brus
        line = re.sub(r"/L\d+[A-Z0-9]*", "", line)

        # ta bort ritningar
        if re.search(r"Ritning\s+\d+", line):
            continue

        # ta bort innehållsförteckning (........)
        if re.search(r"\.{5,}", line):
            continue

        # ta bort rena sidnummer
        if re.fullmatch(r"\d+", line):
            continue

        cleaned_lines.append(line)

    # nu jobbar vi på hela texten
    text = "\n".join(cleaned_lines)

    text = text.replace("-\n", "")   # fixa avstavning
    text = text.replace("\n", " ")   # NU plattar vi ut texten

    text = text.replace("–", "-")
    text = text.replace("—", "-")

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
for doc in documents:
    doc["text"] = clean_text(doc["text"])

## Första chunkningen

In [ ]:
chunks = []
n = 7
overlap = 3

for doc in documents: 
    sentences = re.split(r'(?<=[.!?]) +', doc["text"])

    for i in range(0, len(sentences), n - overlap):
        chunk_sentences = sentences[i:i+n]

        chunk_text = " ".join(chunk_sentences)

        chunks.append({
            "text": chunk_text,
            "source": doc["source"]
        })

In [ ]:
print(len(chunks))

226


## Embeddings

In [ ]:
def create_embeddings(text,
                      model="gemini-embedding-001",
                      task_type="SEMANTIC_SIMILARITY"):
    return client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )

Eftersom jag fick problem med att jag överskred min quota så har jag minskat antal manualer från 5 till 1 samt satt in att spara embeddnings mellan varje lyckad embedding för att inte behöva köra om från början om jag uppnår min quota. Jag satt även in en liten fördröjning mellan chunksen för att minska belastningen

In [ ]:
save_path = "embeddings.json"

# ladda in redan sparade embeddings om filen finns
if os.path.exists(save_path):
    with open(save_path, "r", encoding="utf-8") as f:
        embeddings = json.load(f)    
else:
    embeddings = []

already_done = {item["text"] for item in embeddings}

for chunk in chunks:
    if chunk["text"] in already_done:
        continue

    try:
        response = create_embeddings(chunk["text"])
    
        item = {
        "text": chunk["text"],
        "source": chunk["source"],
        "embedding": response.embeddings[0].values
        }

        embeddings.append(item)

        # spara direkt efter varje lyckad embedding
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(embeddings, f, ensure_ascii=False, indent=2)

        print(f"Sparade chunk från {chunk['source']}")

        time.sleep(1) # liten paus för att minska belastningen

    except Exception as e:
        print("Stopp:", e)
        break

    # Gör om embeddings till numpy‑matris (för beräkning) och metadata‑lista
    chunk_vectors = np.array([item["embedding"] for item in embeddings])
    records = [{"text": item["text"], "source": item["source"]} for item in embeddings]

Stopp: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'EmbedContentRequest.content contains an empty Part.', 'status': 'INVALID_ARGUMENT'}}


## Semantisk sökning funktionerna

In [ ]:
def cosine_similarity(vector, matrix):
    # Normalisera vektorer
    vector_norm = vector / np.linalg.norm(vector)
    matrix_norm = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    # Beräkna similarity för alla på en gång
    return np.dot(matrix_norm, vector_norm.T).flatten()

In [ ]:
def semantic_search(query, top_k=5):
    query_response = create_embeddings(query)
    query_vector = np.array(query_response.embeddings[0].values).reshape(1,-1)

    scores = cosine_similarity(query_vector, chunk_vectors)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "score": float(scores[idx]),
            "text": records[idx]["text"],
            "source": records[idx]["source"]
        })

    return results

## Kontroll av embeddingsen

In [ ]:
def similarity(text_a, text_b):
    emb_a = np.array(create_embeddings(text_a).embeddings[0].values)
    emb_b = np.array(create_embeddings(text_b).embeddings[0].values)
    return cosine_similarity(emb_a, emb_b)


In [ ]:
import numpy as np

def similarity(text_a, text_b):
    emb_a = np.array(create_embeddings(text_a).embeddings[0].values)
    emb_b = np.array(create_embeddings(text_b).embeddings[0].values)

    return np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b))

In [ ]:
pairs = [
    ("Maskinen fylls inte med vatten.", "Apparaten fylls inte med vatten korrekt."),
    ("Rengör maskinen regelbundet.", "Vänta tills nätspänningen är stabil."),
    ("Avbryt ett pågående program.", "Stoppa maskinen medan den kör."),
]

for a, b in pairs:
    print(f"{a}\n{b}\nLikhet: {similarity(a,b):.2f}\n")


Maskinen fylls inte med vatten.
Apparaten fylls inte med vatten korrekt.
Likhet: 0.89

Rengör maskinen regelbundet.
Vänta tills nätspänningen är stabil.
Likhet: 0.75

Avbryt ett pågående program.
Stoppa maskinen medan den kör.
Likhet: 0.86



In [ ]:
pairs = [
    ("Rengör maskinen regelbundet.", "Maskinen har en maximal kapacitet på 8 kg."),
    ("Avbryt ett pågående program.", "Barn ska inte leka med apparaten."),
    ("Maskinen fylls inte med vatten.", "Apparaten fylls inte med vatten korrekt."),
    ("Hunden är ett djur med fyra ben, en svans och den gillar att äta ben.", "Rengör maskinen regelbundet."),
    ("Maskinen fylls inte med vatten.", "Använd flytande tvättmedel för ullprogram.")
]

for a, b in pairs:
    print(f"{a}\n{b}\nLikhet: {similarity(a,b):.2f}\n")

Rengör maskinen regelbundet.
Maskinen har en maximal kapacitet på 8 kg.
Likhet: 0.80

Avbryt ett pågående program.
Barn ska inte leka med apparaten.
Likhet: 0.78

Maskinen fylls inte med vatten.
Apparaten fylls inte med vatten korrekt.
Likhet: 0.89

Hunden är ett djur med fyra ben, en svans och den gillar att äta ben.
Rengör maskinen regelbundet.
Likhet: 0.69

Maskinen fylls inte med vatten.
Använd flytande tvättmedel för ullprogram.
Likhet: 0.77



In [ ]:
def retrieval_evaluation(query):
    results = semantic_search(query)

    for r in results:
        print(f"Score: {r['score']:.3f}")
        print(f"Source: {r['source']}")
        print(r["text"])
        print("-" * 80)

In [ ]:
query = "Hur startar jag maskinen?"

retrieval_evaluation(query)

Score: 0.836
Source: 9000388437_D.pdf
• Ställ programvredet på det nya programmet du vill välja. • Tryck på ”Start/Paus” för att starta. entrifugeringsvarvtalet kan ändras under hela programförloppet. Du kan lägga till och ta bort vilka tilläggsval du vill under hela programmets gång, om dessa är tillåtna i programmet. Avbryta program Om du vill avbryta ett program i förtid: • Ställ programvredet på ”Från”. • Ställ programvredet på ”Centrifugering”. Välj önskat centrifugeringsvarvtal vid ”centrifugering” (ej sköljstopp).
--------------------------------------------------------------------------------
Score: 0.827
Source: 9000388437_D.pdf
Ställa in program Start/paus-knapp Display för programtid och startfördröjning Programvred A 31 Varvtalsväljare Knappar för tilläggsfunktioner Knappen starttid Välja program: Display och inställning av varvtalsvred • Använd programvredet för att välja önskat program. • Använd varvtalsvredet och ställ in önskat centrifugeringsvarvtal, se tabellen på sid

In [ ]:
query = "Hur väljer jag program?"

retrieval_evaluation(query)

Score: 0.869
Source: 9000388437_D.pdf
• Ställ programvredet på det nya programmet du vill välja. • Tryck på ”Start/Paus” för att starta. entrifugeringsvarvtalet kan ändras under hela programförloppet. Du kan lägga till och ta bort vilka tilläggsval du vill under hela programmets gång, om dessa är tillåtna i programmet. Avbryta program Om du vill avbryta ett program i förtid: • Ställ programvredet på ”Från”. • Ställ programvredet på ”Centrifugering”. Välj önskat centrifugeringsvarvtal vid ”centrifugering” (ej sköljstopp).
--------------------------------------------------------------------------------
Score: 0.849
Source: 9000388437_D.pdf
Ställa in program Start/paus-knapp Display för programtid och startfördröjning Programvred A 31 Varvtalsväljare Knappar för tilläggsfunktioner Knappen starttid Välja program: Display och inställning av varvtalsvred • Använd programvredet för att välja önskat program. • Använd varvtalsvredet och ställ in önskat centrifugeringsvarvtal, se tabellen på sid

In [ ]:
query = "Hur rengör jag tvättmedelsfacket?"

retrieval_evaluation(query)

Score: 0.871
Source: 9000388437_D.pdf
Filter i vatteninloppet Vi rekommenderar att filter i vatteninloppet rengörs regelbundet. Tvättmedelsfack Rengör tvättmedelsbehållaren regelbundet. Gör så här: - För att ta loss tvättmedelsfacket trycker du på båda tryckpunkterna samtidigt - Ta loss delarna på fackets bakdel och skilj denna från framdelen - Skölj delarna under rinnande vatten. Töm behållaren helt. - Sätt ihop alla delarna igen i rätt ordningsföljd. Sätt fast tvättmedelsfacket under maskinens lucka igen. Rengöring av filtret askineffekten kan påverkas negativt om pumpfiltret inte rengörs regelbundet.
--------------------------------------------------------------------------------
Score: 0.862
Source: 9000388437_D.pdf
Använd inte: • Skurpulver • Metall- eller plastsvampar • Alkoholbaserade produkter (sprit, lösningsmedel etc.) Inre delar Vi rekommenderar att: • Låt luckan stå öppen en stund efter avslutat tvätt. • Rengör alla plast- och gummidetaljer i trumöppningen med tvålvatten en

In [ ]:
query = "Hur ofta ska jag rengöra maskinen?"

retrieval_evaluation(query)

Score: 0.825
Source: 9000388437_D.pdf
Filter i vatteninloppet Vi rekommenderar att filter i vatteninloppet rengörs regelbundet. Tvättmedelsfack Rengör tvättmedelsbehållaren regelbundet. Gör så här: - För att ta loss tvättmedelsfacket trycker du på båda tryckpunkterna samtidigt - Ta loss delarna på fackets bakdel och skilj denna från framdelen - Skölj delarna under rinnande vatten. Töm behållaren helt. - Sätt ihop alla delarna igen i rätt ordningsföljd. Sätt fast tvättmedelsfacket under maskinens lucka igen. Rengöring av filtret askineffekten kan påverkas negativt om pumpfiltret inte rengörs regelbundet.
--------------------------------------------------------------------------------
Score: 0.823
Source: 9000388437_D.pdf
Använd inte: • Skurpulver • Metall- eller plastsvampar • Alkoholbaserade produkter (sprit, lösningsmedel etc.) Inre delar Vi rekommenderar att: • Låt luckan stå öppen en stund efter avslutat tvätt. • Rengör alla plast- och gummidetaljer i trumöppningen med tvålvatten en

In [ ]:
query = "Vad ska jag tänka på innan jag startar maskinen?"

retrieval_evaluation(query)

Score: 0.834
Source: 9000388437_D.pdf
- Kontrollera att tvättmaskinen står stadigt. yte av nätkabel Av säkerhetsskäl får nätkabeln bara bytas av vår auktoriserade kundtjänst. Internationella skötselsymboler ill hjälp vid skötsel av textilier innehåller dessa en etikett med den nödvändiga informationen. Denna visas med följande symboler: © COFREET Före den första tvätten - kör maskinen en gång utan tvätt Innan du tvättar för första gången rekommenderar vi att du kör programmet ”VITTVÄTT 90 °C utan förtvätt” med ett halvt mått tvättmedel. På så vis får du bort eventuella rester av testvatten som kan finnas kvar i maskinen. Punkterna betecknar torkstegen i torktumlaren. Bokstäverna avser de kemiska rengöringsmedlen.
--------------------------------------------------------------------------------
Score: 0.828
Source: 9000388437_D.pdf
•Fyll på med stärkelselösningen i facket. •Tryck på knappen “Start/Paus”. Färgning/Avfärgning Färgning ska endast ske för vardagsbruk. Salt kan angripa rostfr

In [ ]:
query = "Kan jag lämna luckan stängd efter tvätt?"

retrieval_evaluation(query)

Score: 0.841
Source: 9000388437_D.pdf
- efter programslut: Om displayen visar kan du välja ett nytt program utan att först ställa programvredet på ”Från”.1 A Säkerhetsfunktioner Lockets öppningssäkring: När du startar programmet spärras luckan. Luckspärren öppnas efter avslutat program och i läget ”sköljstopp”. På vissa modeller: Om du programmerar fördröjd start så är luckan spärrad under hela startfördröjningstiden. Om du vill öppna luckan under pågående program, tryck kortvarigt på ”Start/Paus” och vänta en till två minuter tills luckspärren öppnas. ör att undvika brännskador är spärren aktiv tills tvättvattnet har svalnat något. Vattensäkring: Under driften förebyggs eventuell översvämning av den ständiga vattennivåkontrollen. Centrifugeringssäkerhet: Tvättmaskinen är försedd med en säkerhetsfunktion som kan minska centrifugeringen om tvätten är felaktigt fördelad.
--------------------------------------------------------------------------------
Score: 0.835
Source: 9000388437_D.pdf

In [ ]:
query = "Hur undviker jag dålig lukt?"

retrieval_evaluation(query)

Score: 0.787
Source: 9000388437_D.pdf
SMÖRJMEDEL - TJÄRA: Använd fläckborttagningsmedel*. Alternativt kan du lägga smör på fläcken, låta det verka och sedan suga upp det med terpentinolja*. FÄRGER: Låt inte färgfläckar torka, utan behandla dem genast med det lösningsmedel som anges på färgförpackningen (vatten, terpentin*, lättbensin*). Smörj in och skölj ur. STEARIN: Skrapa bort större delen av fläcken. Lägg sedan hushållspapper på båda sidor om tyget och smält resten av stearinet med hjälp av strykjärn. MAKE-UP: Lägg den fläckiga tygsidan på en bit hushållspapper och fukta tygets baksida med 90-procentig alkohol.
--------------------------------------------------------------------------------
Score: 0.781
Source: 9000388437_D.pdf
Vid frostrisk måste slangen för vattentillförsel kopplas loss och allt vatten i maskinen tömmas ut. På så vis tappas det överskottsvatten ut som kan finnas kvar i behållaren. Hölje Använd endast vatten och tvål. Manöverpanel, sockel, osv.: Använd endast fukt

Efter första utvärderingen (n=10 och overlap=2) så ser jag att embeddingen verkar fungera bra men chunksen är för stora då det kommer med ganska mycket orelevant information också. 
Så jag väljer att ändra till n=5 men behåller overlap=2.

Efter andra utvärderingen (n=5 och overlap=2) så blir retrivaln mycket sämre. Frågorna om vattnet tappade hittar mest orelevanta saker. Detta kan vara för att problemet och lösningen hamnar i olika chunks. Så mitt nästa test blir att öka overlap till 3 för att se vad det ger för resultat. 

Efter tredje utvärderingen (n=5 och overlap=3) så ser jag att det blir bättre men fortfarande inte bra, det fattas forfarande lite info, särskilt i de sam handlar om vatten. Sen ser jag också att vi får in massa av punkter i några av chunksen vilket jag inser är innehållsförteckningen. Så jag väljer att modifiera inläsningen så att jag inte läser in framsidan och de första orelevanta sidorna. Och mitt nästa test blir n=7 men behåller overlap=3.

Efter fjärde utvärderingen (n=7 och overlap=3) så ser det hyftast ut men jag inser också att jag ställer fel frågor för retrivlen då jag fokuserar förmycket på felsökning (en efterdyning från min första idé som var att göra en chatbot till servicetekniker ute i fältet) och manualen jag använder har ingen större felsökningsdel utan den handlar mer om hur man använder sig utav maskinen. Så efter att jag ändrade alla frågor till mer relevanta frågor så fick jag mycket bättre resultat och jag är nu redo att gå vidare till att göra själva chatboten.

# Chatboten

In [ ]:
system_prompt = """Jag kommer ställa dig en fråga, och jag vill att di svarar baserat bara på kontexten jag skickar med, och ingen annan information. 
Om det inte finns nog med information i kontexten för att svara på frågan, säg "Det vet jag inte". Försök inte att gissa. 
Formulera dig enkelt och dela upp svaret i fina stycken. """

In [ ]:
def generate_user_prompt(query):
    context = "\n".join(semantic_search(query, chunks, embeddings))

    user_prompt = f"Frågan är {query}. Här är kontexten: {context}"

    return user_prompt

In [ ]:
def generate_response(system_prompt, user_message, model="gemini-2.5-flash"):
    response = client.models.generate_content(
        model=model,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt),
            contents=generate_user_prompt(user_message)
        )
    return response

In [ ]:
print(generate_response(system_prompt, "Hur får jag bort färgfläckar på kläderna?").text)